Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo

%run "plot_style_kalinka.py"


#### Process Run

In [ ]:
df_event_rate = pd.read_csv(
    # "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/peak_all_channel_event_rate_20250811_145300.csv",
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/peak_all_channel_event_rate_20250812_133321.csv",
    parse_dates=["absolute_time"]
    )

In [ ]:
df_important = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/important_time_stamp.csv", 
    parse_dates=["date_time"]
    )

In [ ]:
plt.hist(df_event_rate.integral_area_Vns_board_channel0, bins = 100)
#logy

plt.yscale('log')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))


mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")) & (
    df_event_rate.board == 0) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)
ax.hist(df_event_rate.absolute_time[mask], bins=1500)

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            


            
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)


In [ ]:
df_event_rate.columns

In [ ]:
df_event_rate.runtime_s

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
    df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 0)
tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
runtime_s = tritium_data.groupby(['md_full_path'])['runtime_s'].mean().to_numpy()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    
    # ax.scatter(start_time[mask], threshold_adc[mask], s = area[mask], color='green')
# ax.plot(start_time, event_rate, '-')


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 10)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()
runtime_s = tritium_data.groupby(['md_full_path'])['runtime_s'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 10)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()
runtime_s = tritium_data.groupby(['md_full_path'])['runtime_s'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / runtime_s

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
start_time = tritium_data['absolute_time'].min()
end_time = tritium_data['absolute_time'].max()
nbins = int((end_time - start_time).total_seconds())
nbins

In [ ]:



mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_PE_board_channel0 > 10)

tritium_data = df_event_rate[mask]
start_time = tritium_data['absolute_time'].min()
end_time = tritium_data['absolute_time'].max()
nbins = int((end_time - start_time).total_seconds())


# convert time stamp to unix time
unix_time = tritium_data['absolute_time'].astype('int64') // 10**9

counts, bin_edges = np.histogram(unix_time, bins = nbins)

# Compute the event rate
event_rate = counts / np.diff(bin_edges)

# average every 100 entries
average_length = 10
remainder_length = len(event_rate) % average_length
event_rate = np.mean(event_rate[:-remainder_length].reshape(-1,average_length),axis=1)

# convert unix time to time stamp
time_stamps = pd.to_datetime(bin_edges[:-remainder_length:average_length][:-1], unit='s')


#

fig, ax = plt.subplots(figsize=(15, 6))

ax.scatter(time_stamps, event_rate)

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("5.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

area = df_event_rate[mask].groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()


plt.hist(area)

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        # df_event_rate.integral_area_Vns_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_PE_board_channel0 > 2.5)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_PE_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()
mean_baseline_std_V = tritium_data.groupby(['md_full_path'])['mean_baseline_std_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")

    ax2.scatter(start_time[mask], mean_baseline_std_V[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V", color = 'green')

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


##### No difference between board 0 and 1 and all -> go to board 0 channel 0 only

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")

    ax2.scatter(start_time[mask], threshold_adc[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V", color = 'green')

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")

    ax2.scatter(start_time[mask], threshold_adc[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V", color = 'green')

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 100) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))
ax2 = ax.twinx()


ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Area', fontsize=12)

In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 10) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 180, 
            f"{comment}", 
            rotation=45, 
            ha='left', 
        #     fontsize=12, 
            color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

#format date time axis
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Integral Area [PE]')

ax.set_xlabel('Time')
ax.set_ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_PE_board_channel0 > 20) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 180, 
            f"{comment}", 
            rotation=45, 
            ha='left', 
        #     fontsize=12, 
            color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

#format date time axis
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Integral Area [PE]')

ax.set_xlabel('Time')
ax.set_ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(1,4, figsize=(20, 7), sharey=True)


run_tag_list = ["LXe/gain_calibration", "LXe/Cs137", "LXe/Co57", "LXe/tritium"]

for i, run_tag in enumerate(run_tag_list):
        mask = (df_event_rate.run_tag == run_tag) & (
                # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
                df_event_rate.integral_area_PE_board_channel0 > 10) & (
                (df_event_rate.voltage_preamp1_V <= -46))

        data = df_event_rate[mask]

        number_of_events = data.groupby(['md_full_path'])['board'].count().to_numpy()
        start_time = data.groupby(['md_full_path'])['absolute_time'].min()
        end_time = data.groupby(['md_full_path'])['absolute_time'].max()
        area = data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
        threshold_adc = data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
        voltage_preamp1_V = data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


        delta_time = (end_time - start_time).dt.total_seconds()
        event_rate = number_of_events / delta_time

        ax[i].scatter(start_time, event_rate, c = area)
        ax[i].set_title(run_tag, fontsize=16)
        #format date time axis
        ax[i].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))



# for date_time, comment in zip(df_important.date_time, df_important.comment):
#     ax[3].axvline(date_time, color='r', linestyle="dashed")
#     ax[3].text(date_time, 180, 
#             f"{comment}", 
#             rotation=45, 
#             ha='left', 
#         #     fontsize=12, 
#             color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

# color bar
cbar = plt.colorbar(ax[3].collections[0], ax=ax[3], orientation='vertical')
cbar.set_label('Integral Area [PE]')

plt.xlabel('Time')
plt.ylabel('Event Rate [Hz]')

# plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 30) & (
        df_event_rate.integral_area_Vns_board_channel0 < 40) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        df_event_rate.board == 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(start_time, event_rate, c = area)


for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 180, 
            f"{comment}", 
            rotation=45, 
            ha='left', 
        #     fontsize=12, 
            color='black')
    
    

# rotate x axis
plt.xticks(rotation=30)

#format date time axis
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d %H:%M'))

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Integral Area [PE]')

ax.set_xlabel('Time')
ax.set_ylabel('Event Rate [Hz]')

plt.savefig('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/peak_all_channel_event_rate.pdf', dpi=100, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))

for threshold_sig in np.arange(5, 11):
        mask = util.vec_regex_search(f"{int(threshold_sig)}.0sig", df_event_rate.md_full_path) & (
        df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0) & (
        df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

        tritium_data = df_event_rate[mask]

        number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
        start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
        end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
        area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
        threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
        voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()

        delta_time = (end_time - start_time).dt.total_seconds()
        event_rate = number_of_events / delta_time

        ax.scatter(start_time, event_rate, s=area, label = f"{threshold_sig} sig")

for date_time, comment in zip(df_important.date_time, df_important.comment):
        ax.axvline(date_time, color='r', linestyle="dashed")
        ax.text(date_time, 0.1, 
                f"{comment}", 
                rotation=90, 
                ha='left', 
                fontsize=12, 
                color='black')

# rotate x axis
plt.xticks(rotation=45)

# color bar
# cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
# cbar.set_label('Area', fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)

In [ ]:
mask = (
    # df_event_rate.absolute_time >= pd.Timestamp("2024-10-30 00:00:00")) & (
        # util.vec_regex_search("10.0sig", df_event_rate.md_full_path)) & (
        # df_event_rate.voltage_preamp1_V == -46) & (
        # df_event_rate.board == 1) & (
        df_event_rate.integral_area_Vns_board_channel0 > 0)

tritium_data = df_event_rate[mask]

number_of_events = tritium_data.groupby(['md_full_path'])['board'].count().to_numpy()
start_time = tritium_data.groupby(['md_full_path'])['absolute_time'].min()
end_time = tritium_data.groupby(['md_full_path'])['absolute_time'].max()
area = tritium_data.groupby(['md_full_path'])['integral_area_Vns_board_channel0'].mean().to_numpy()
threshold_adc = tritium_data.groupby(['md_full_path'])['threshold_adc'].mean().to_numpy()
voltage_preamp1_V = tritium_data.groupby(['md_full_path'])['voltage_preamp1_V'].mean().to_numpy()


delta_time = (end_time - start_time).dt.total_seconds()
event_rate = number_of_events / delta_time

fig, ax = plt.subplots(figsize=(20, 4))

for voltage in np.unique(voltage_preamp1_V):
    mask = voltage_preamp1_V == voltage
    ax.scatter(start_time[mask], event_rate[mask], s = area[mask], 
               label=f"Voltage: {voltage:.2f} V")
    

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)

plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=12)


In [ ]:
plt.hist(area)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))

mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")) & (df_event_rate.board == 1) & (df_event_rate.integral_area_Vns_board_channel0 > 0)
ax.hist(df_event_rate.absolute_time[mask], bins=1500)

for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)


#### Average every 100 seconds

In [ ]:
tritium_time = df_event_rate.absolute_time[df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")]

event_max = np.max(tritium_time)
event_min = np.min(tritium_time)
n_bins = int((event_max - event_min).total_seconds()) # 1 second per bin


In [ ]:
mask = df_event_rate.absolute_time >= pd.Timestamp("2024-10-30")
#convert pd timestamp to unix time
unix_time = df_event_rate['absolute_time'][mask].astype(np.int64) // 10**9

event_rate, event_time = np.histogram(unix_time, bins = n_bins)

# convert unix time to pd timestamp
event_time = pd.to_datetime(event_time, unit='s')

# average event rate for every 100 entries

# remove last few entries to make it divisible by 100
tmp = len(event_rate)%100
event_rate = event_rate[:-tmp]

n_bins_after_average = len(event_rate) // 100

event_rate = event_rate.reshape(-1, 100).mean(axis=1)
event_time = event_time[::100][:-1]

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))


ax.scatter(event_time, event_rate)
for date_time, comment in zip(df_important.date_time, df_important.comment):
    ax.axvline(date_time, color='r', linestyle="dashed")
    ax.text(date_time, 0.1, 
            f"{comment}", 
            rotation=90, 
            ha='left', 
            fontsize=12, 
            color='black')

# rotate x axis
plt.xticks(rotation=45)


In [ ]:
df_event_rate.columns

In [ ]:
mask = np.isnan(df_event_rate.integral_area_PE_board_channel0)

len(df_event_rate.integral_area_PE_board_channel0[mask])

In [ ]:
len(df_event_rate.integral_area_PE_board_channel0)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))

mask = (df_event_rate.absolute_time >= pd.Timestamp("2024-10-30"))

ax.scatter(df_event_rate.absolute_time[mask], df_event_rate.integral_area_Vns_board_channel0[mask], s=1)

# for date_time, comment in zip(df_important.date_time, df_important.comment):
#     ax.axvline(date_time, color='r', linestyle="dashed")
#     ax.text(date_time, 0.1, 
#             f"{comment}", 
#             rotation=90, 
#             ha='left', 
#             fontsize=12, 
#             color='black')

# rotate x axis
plt.xticks(rotation=45)


In [ ]:
plt.hist(df_event_rate.absolute_time[df_event_rate.absolute_time > pd.Timestamp("2024-10-25")], bins=100)

# rotate x axis
plt.xticks(rotation=45)


In [ ]:
mask = ((df_event_rate["integral_area_PE_board_channel0"])>0)
df_event_rate[mask]

In [ ]:
df_event_rate.board0_time

In [ ]:
mask = (pd.Timestamp(df_event_rate.board0_time) < pd.Timestamp('2024-11-02 00:01'))
plt.plot(df_event_rate.board0_time[mask], label='Board 0')
mask = (df_event_rate.board1_time < pd.Timestamp('2024-11-02 00:01'))
plt.plot(df_event_rate.board1_time[mask], label='Board 1')
plt.xlabel("Event number")
plt.ylabel("Event time")

# format time axis to MM-DD HH:MM:SS
plt.gca().yaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%m-%d %H:%M:%S'))


plt.legend()

In [ ]:
plt.plot(all_runs_d2d.date_time)


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))



event_max = np.max(board0_time)
event_min = np.min(board0_time)
n_bins = int((event_max - event_min).total_seconds()) # 1 second per bin

# ax.hist(board0_time, range = [event_min,event_min + pd.Timedelta(n_bins,'s')], bins = n_bins, alpha=0.5)
mask = board0_time < pd.Timestamp("2024-11-02 00:00:00")
ax.hist(board0_time[mask], bins = 200, alpha=0.5)
mask = board1_time < pd.Timestamp("2024-11-02 00:00:00")
ax.hist(board1_time[mask], bins = 200, alpha=0.5)

# rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')
# time formatting to MM-DD HH:MM:SS
ax.xaxis.set_major_formatter(
    plt.matplotlib.dates.DateFormatter('%m-%d %H:%M:%S')
)

ax.axvline(pd.Timestamp("2024-10-30 17:37:15"), color='red', linestyle='--', label='stable detector condition')
ax.axvline(pd.Timestamp("2024-10-30 18:56:35"), color='red', linestyle='--', label='stable detector condition')

# log y
ax.set_yscale('log')

ax.set_xlabel('Time [s]')
ax.set_ylabel('Event Rate [Hz]')

plt.show()

In [ ]:
### Choose a run from the list above
# initialize the result storage
df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
single_info_list = []

# for md_full_path in all_run_list[run_id:run_id+1]:
for md_full_path in all_run_list:
    result, waveform, baseline, baseline_std = get_peak_level_data(
        all_runs_d2d=all_runs_d2d,
        md_full_path=md_full_path,
        peak_merge_window_sample=250
    )
    single_info_list += result

df_result = pd.DataFrame.from_dict(single_info_list)
d2d_data = d2d.data(df_result)

d2d_data.integral_window_area_PE
array = list(d2d_data.integral_window_area_PE)
d2d_data.integral_window_area_PE = np.concatenate(array)


In [ ]:
### Choose a run from the list above
# initialize the result storage
df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
single_info_list = []

# for md_full_path in all_run_list[run_id:run_id+1]:
for md_full_path in all_run_list:
    result, waveform, baseline, baseline_std = get_peak_level_data(
        all_runs_d2d=all_runs_d2d,
        md_full_path=md_full_path,
        peak_merge_window_sample=250
    )
    single_info_list += result

df_result = pd.DataFrame.from_dict(single_info_list)
d2d_data = d2d.data(df_result)

d2d_data.integral_window_area_PE
array = list(d2d_data.integral_window_area_PE)
d2d_data.integral_window_area_PE = np.concatenate(array)


In [ ]:
# df = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/co57_47V.csv",
#     parse_dates=["date_time"],
#         delimiter=",",
#         quotechar='"', 
#         skipinitialspace=True, 
#         encoding="utf-8")

# d2d_data = d2d.data(df)

### Check Results (Plots)

In [ ]:
# data selection
# mask = (d2d_data.peak_area_PE > 1.5) 
# mask = (d2d_data.peak_height_V < 1.2) 
# mask = (d2d_data.peak_width_ns < 300) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500)
# mask = (d2d_data.peak_width_ns > 300)
# selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
selected_data = d2d_data

plt.close()
fig_peak_integral_area, axes_peak_integral_area = plt.subplots(3,8,figsize=(35,15))
fig_area_height, axes_area_height = plt.subplots(3,8,figsize=(35,15))
fig_area_width, axes_area_width = plt.subplots(3,8,figsize=(35,15))
fig_area_height_full, axes_area_height_full = plt.subplots(3,8,figsize=(35,15))
fig_area_width_full, axes_area_width_full = plt.subplots(3,8,figsize=(35,15), sharex=True, sharey='col')
fig_width_height, axes_width_height = plt.subplots(3,8,figsize=(35,15))

axes = [axes_area_height, axes_area_height_full, axes_area_width, axes_area_width_full, axes_width_height, axes_peak_integral_area]
figures = [fig_area_height, fig_area_height_full, fig_area_width, fig_area_width_full, fig_width_height, fig_peak_integral_area] 
axes_name = ['axes_area_height', 
            'axes_area_height_full', 
            'axes_area_width', 
            'axes_area_width_full', 
            'axes_width_height', 
            'axes_peak_integral_area']



for channel in range(24):
    mask = selected_data.channel == channel
    singl_channel_data = selected_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V
    area_PE = singl_channel_data.peak_area_PE
    width_ns = singl_channel_data.peak_width_ns
    integral_window_area_PE = list(singl_channel_data.integral_window_area_PE)
    integral_window_area_PE = np.concatenate(integral_window_area_PE)

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes_peak_integral_area[plot_row,plot_col].hist((area_PE),
                            bins=100,
                            range=[-0.1,10], 
                            alpha=0.5,
                            label='peak_area_PE_array')
    axes_peak_integral_area[plot_row,plot_col].hist((integral_window_area_PE),
                            bins=100,
                            range=[-0.1,10], 
                            alpha=0.5,
                            label='integral_window_area_PE')   
    axes_peak_integral_area[plot_row,plot_col].set_yscale('log')
    if plot_row == 2:
        axes_peak_integral_area[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_peak_integral_area[plot_row,plot_col].set(ylabel='Counts')

    axes_area_height[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,10],[0,0.1]],
                            cmap='viridis',
                            norm=LogNorm())
    if plot_row == 2:
        axes_area_height[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_height[plot_row,plot_col].set(ylabel='Height [V]')


    axes_area_width[plot_row,plot_col].hist2d(area_PE,width_ns,
                            bins=[100,100],
                            range=[[-0.1,10],[0,1000]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_area_width[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_width[plot_row,plot_col].set(ylabel='Width [ns]')

    
    axes_area_height_full[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,1500],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())
    if plot_row == 2:
        axes_area_height_full[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_height_full[plot_row,plot_col].set(ylabel='Height [V]')


    axes_area_width_full[plot_row,plot_col].hist2d(area_PE,width_ns,
                            bins=[50,50],
                            range=[[-0.1,1000],[0,2500]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_area_width_full[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_width_full[plot_row,plot_col].set(ylabel='Width [ns]')
    
    
    axes_width_height[plot_row,plot_col].hist2d(width_ns,height_V,
                            bins=[50,50],
                            range=[[0,1000],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_width_height[plot_row,plot_col].set(xlabel='Width [ns]')
    if plot_col == 0:
        axes_width_height[plot_row,plot_col].set(ylabel='Height [V]')
    
    for ax in axes:
        ax[plot_row,plot_col].set_title(f"Channel {channel}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend(axes_peak_integral_area, loc='upper right', fontsize='small')
axes_peak_integral_area[2,7].legend(loc="upper right")

# Set plot title
for fig, ax_name in zip(figures, axes_name):
    short_name = selected_data.md_full_path[0][0].replace('/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/','')
    short_name = short_name.replace('/','_')

    fig.suptitle(f"{ax_name}: \nPath: {short_name}\nComment: {selected_data.comment[0][0]}\nRun tag: {selected_data.run_tag[0][0]}\nVoltage: {selected_data.voltage_preamp1_V[0][0]}",
                 y=1.02, x=0.1,ha='left')
    
    
    os.makedirs(short_name, exist_ok=True)
    fig.savefig(f"{short_name}/{ax_name}.png", dpi=300, bbox_inches='tight')

In [ ]:
plt.close()
fig, axes = plt.subplots(3,8,figsize=(35,15))

# masking
for channel in range(24):
    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes[plot_row,plot_col].hist(height_V,
                            bins=100,
                            range=[0,0.1])
    
    axes[plot_row,plot_col].set_title(f"Channel {channel}")

# set the labels
for ax in axes.flat:
    ax.set(xlabel='Height [V]', ylabel='Count')
    # log y
    ax.set_yscale('log')
    # ax.set_xlim(-0.1, 100)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])


plt.show()

### Check waveform

In [ ]:
np.unique(d2d_data.DC_OFFSET)

In [ ]:
for channel in range(24):
    mask = d2d_data.channel == f"[{channel}]"
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    height_V = singl_channel_data.peak_height_V

    plt.hist(singl_channel_data.peak_height_V, alpha = 0.5, bins=100, range=[1,1.5], label=f"Channel {channel}")

plt.xlabel("Peak Height [V]")
plt.ylabel("Counts")
plt.legend(ncols=2, loc='upper right', fontsize='small')

In [ ]:
mask = (d2d_data.peak_height_V > 1.1) & (d2d_data.channel == f"[{14}]")
# mask = (d2d_data.peak_area_PE > 1e100) 
# mask = (d2d_data.peak_width_ns > 300) & (d2d_data.peak_height_V > 0.4) & (d2d_data.channel == 14)
# mask = (d2d_data.peak_width_ns > 400) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500) & (d2d_data.channel == 11)
# mask = (d2d_data.peak_height_V > 1.2) & (d2d_data.channel == 11)
selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
len(selected_data)

single_data = selected_data.get_row_info_to_dict(0)
single_info = WaveformInfo()
single_info.set_info_from_dict(single_data)
waveform, baseline, baseline_std = get_waveform_from_single_info(single_info)

event_id_array = selected_data.event_id


In [ ]:
# can specify the event_id here
# event_id = 6564
event_id = None

if event_id is None:
    event_id_id = np.random.randint(0, len(event_id_array), size=1)[0]
    event_id = event_id_array[event_id_id]

print(f"Selected event_id: {event_id}")

single_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=3
extend_sum_window=50
peak_merge_window_sample = 250 # samples
        
window_size = 6
min_peak_width_sample = window_size*3

single_info.set_peaks_for_single_processed_waveform(
                    single_waveform, 
                    single_baseline, 
                    single_baseline_std,
                    threshold_sig=5, 
                    extend_sum_window=50,
                    peak_merge_window_sample=peak_merge_window_sample,
                    show_plot = True,
                    event_id=event_id
                )


fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/example_waveform-{event_id}.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')


In [ ]:
peak_boundaries

In [ ]:
mask = selected_data.event_id == event_id
test = selected_data.apply_mask(mask, inplace=False, dry=True)

print(test.peak_area_PE)

In [ ]:
result, edge = np.histogram(d2d_data.peak_height_V, bins = 100, range=[0,0.1])
edge[np.argmax(result)]  # get the bin center of the peak

### Event Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
# this also remove null values from the peak_height_V_array
array = list(d2d_data.peak_start_time_s_array)
peak_start_time = np.concatenate(array)

event_max = np.max(peak_start_time)
event_min = np.min(peak_start_time)
n_bins = int(event_max - event_min) # 1 second per bin

ax.hist(peak_start_time, range = [event_min,event_min+n_bins], bins = n_bins, alpha=0.5)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Event Rate [Hz]')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
n_bins = 207


array = list(d2d_data.peak_rel_start_time_s_array)
peak_start_time = np.concatenate(array)
ax.hist(peak_start_time*1e6, alpha=0.5, bins = n_bins,
        label='without merge window'
        )

# this also removes null values from the peak_height_V_array
array = list(d2d_data_2.peak_rel_start_time_s_array)
peak_start_time = np.concatenate(array)
ax.hist(peak_start_time*1e6, alpha=0.5, bins = n_bins,
        label='with 1 us merge window'
        )

ax.set_xlim(0, 4)  # limit x-axis to 1000 us

# log y
ax.set_yscale('log')

ax.set_xlabel('Time [us]')
ax.set_ylabel('Counts')

plt.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
# this also remove null values from the peak_height_V_array
array = list(d2d_data_nomerge.peak_rel_start_time_s_array)
peak_rel_start_time = np.concatenate(array)

array = list(d2d_data_nomerge.peak_area_Vns_array)
peak_area = np.concatenate(array)

# event_max = np.max(peak_start_time)
# event_min = np.min(peak_start_time)
# n_bins = int(event_max - event_min) # 1 second per bin
# ax.hist2d(peak_rel_start_time*1e6, peak_area, bins = n_bins,
#         # range = [event_min,event_min+n_bins], bins = n_bins
#         )

ax.hist2d(peak_rel_start_time*1e6, np.log10(peak_area),
        bins=[100,100],
        # range=[[-0.1,1000],[0,2500]],
        cmap='viridis',
        norm=LogNorm()
        ) 

ax.axhline(0)


ax.set_xlabel('Time [us]')
ax.set_ylabel('log(Area [PE])')



plt.show()

In [ ]:
event_rate,bin_edge = np.histogram(peak_start_time, range = [event_min,event_min+n_bins], bins = n_bins)

### Estimate Coincidence Window

In [ ]:
DetectorHeight = 100 #mm
DetectorDiameter = 100 #mm
DetectorLongestLength = (DetectorHeight**2 + DetectorDiameter**2)**0.5

SpeedofLight = 299792458 # m/s
CoincidenceTime = DetectorLongestLength / SpeedofLight * 1e9 # in ns

In [ ]:
CoincidenceTime

### Attempt to find the Time difference between board 0 and 1

In [ ]:
mask = d2d_data.board == 0
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

board_0_peak_time = single_board_data.peak_rel_start_time_s

mask = d2d_data.board == 1
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
board_1_peak_time = single_board_data.peak_rel_start_time_s

length = np.min([len(board_0_peak_time), len(board_1_peak_time)])
time_diff = np.abs(board_1_peak_time[:length] - board_0_peak_time[:length]) # in s

# plt.hist(time_diff, bins=50, range = [0, 1e-7])
plt.hist(time_diff, bins=50, range=[1.5e-7,0.5e-6])
plt.show()


In [ ]:
count, edge = np.histogram(time_diff, bins=50, range=[1.5e-7,0.5e-6])
edge[np.argmax(count)]  # get the bin center of the peak

### Area Spectrum

In [ ]:
# @jit(nopython=True)
def get_sum_area_PE_in_time_window(
    peak_start_time_s: np.ndarray, 
    relative_start_time: np.ndarray, 
    event_time: np.ndarray, 
    channel: np.ndarray, 
    board: np.ndarray, 
    peak_area_PE: np.ndarray,
    event_id: int,
    time_window_width_s: float,
    coincidence: int,
    ):
    """
    Calculate the sum of peak area for coincidental signals within a specified time window.
    
    Parameters:
    """
    counts = 0
    max_bin_edge = np.max(peak_start_time_s)
    min_bin_edge = np.min(peak_start_time_s)
    n_bins = int((max_bin_edge - min_bin_edge)/time_window_width_s) 
    event_in_window,bin_edge = np.histogram(peak_start_time_s, range = [min_bin_edge,max_bin_edge], bins = n_bins)
    
    sum_area_PE_list = []
    rel_time = []
    mask = event_in_window > coincidence
    start_time_window = bin_edge[:-1][mask]
    end_time_window = start_time_window + time_window_width_s

    for start_time, end_time in zip(start_time_window, end_time_window):
        mask = peak_start_time_s >= start_time
        mask &= peak_start_time_s < end_time
        channels_within_window = channel[mask]
        board_within_window = board[mask]
        area_PE_within_window = peak_area_PE[mask]
        event_id_within_window = event_id[mask]
        event_time_s_within_window = event_time[mask]
        rel_time_within_window = relative_start_time[mask]
        
        if (len(np.unique(board_within_window)) == 2) & (len(np.unique(channels_within_window)) >= coincidence):
            # if (len(np.unique(channels_within_window)) >= coincidence):
            sum_area_PE = np.sum(area_PE_within_window)
            sum_area_PE_list.append(sum_area_PE)
            rel_time.append(np.min(event_time_s_within_window))
            
    return (sum_area_PE_list, rel_time)



In [ ]:
# @jit(nopython=True)
def get_counts_in_time_window(
    peak_start_time_s: np.ndarray, 
    relative_start_time: np.ndarray, 
    event_time: np.ndarray, 
    channel: np.ndarray, 
    board: np.ndarray, 
    peak_area_PE: np.ndarray,
    event_id: int,
    time_window_width_s: float,
    coincidence: int,
    ):
    """
    Calculate the sum of peak area for coincidental signals within a specified time window.
    
    Parameters:
    """
    counts = 0
    max_bin_edge = np.max(peak_start_time_s)
    min_bin_edge = np.min(peak_start_time_s)
    n_bins = int((max_bin_edge - min_bin_edge)/time_window_width_s) # 1 second per bin
    event_in_window,bin_edge = np.histogram(peak_start_time_s, range = [min_bin_edge,max_bin_edge], bins = n_bins)
    
    mask = event_in_window > coincidence
    start_time_window = bin_edge[:-1][mask]
    end_time_window = start_time_window + time_window_width_s

    for start_time, end_time in zip(start_time_window, end_time_window):
        mask = peak_start_time_s >= start_time
        mask &= peak_start_time_s < end_time
        channels_within_window = channel[mask]
        board_within_window = board[mask]
        
        if (len(np.unique(board_within_window)) == 2) & (len(np.unique(channels_within_window)) >= coincidence):
            counts += 1
            
    return counts



Attempt to find baoard 1 and board 0 delay time

In [ ]:
# change peak_start_time_s for all board 1 in d2d_data
count_list = []
delay_time_list = np.arange(-10,10,1)

for delay_time in delay_time_list:
    change_time = d2d_data.get_df()
    change_time.loc[change_time.board == 1, 'peak_start_time_s'] += delay_time
    d2d_data_updated = d2d.data(change_time)
    mask = (d2d_data_updated.peak_area_PE > d2d_data_updated.spe_position*1.5)
    clean_data = d2d_data_updated.apply_mask(mask, inplace=False, dry=True)

    # mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
    # clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # sum_area_PE_list_0, rel_time_0 = get_sum_area_PE_in_time_window(
    counts = get_counts_in_time_window(
        clean_data.peak_start_time_s, 
        clean_data.peak_rel_start_time_s,
        clean_data.event_start_time_s, 
        clean_data.channel, 
        clean_data.board, 
        clean_data.peak_area_PE, 
        clean_data.event_id,
        time_window_width_s = 1,  # 1 us
        coincidence=2)
    count_list.append(counts)


In [ ]:
delay_time_list[np.argmax(count_list)]  # get the delay time with the maximum count

In [ ]:
plt.scatter(delay_time_list, count_list)

#### Summed spectrum

In [ ]:
# delay_time = -1.9
delay_time = 0

change_time = d2d_data.get_df()
change_time.loc[change_time.board == 1, 'peak_start_time_s'] += delay_time
d2d_data_updated = d2d.data(change_time)

mask = (d2d_data_updated.peak_area_PE > d2d_data_updated.spe_position*1.5) 
# & (d2d_data_updated.peak_height_V < 1.2)
clean_data = d2d_data_updated.apply_mask(mask, inplace=False, dry=True)

# mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
# clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

sum_area_PE_list, rel_time = get_sum_area_PE_in_time_window(
    clean_data.peak_start_time_s, 
    clean_data.peak_rel_start_time_s,
    clean_data.event_start_time_s, 
    clean_data.channel, 
    clean_data.board, 
    clean_data.peak_area_PE, 
    clean_data.event_id,
    time_window_width_s = 0.0000001,  # 1 us
    coincidence=3)

In [ ]:

mask = (d2d_data_2.peak_area_PE > d2d_data_2.spe_position*1.5) 
# & (d2d_data_updated.peak_height_V < 1.2)
clean_data = d2d_data_2.apply_mask(mask, inplace=False, dry=True)

# mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
# clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

sum_area_PE_list_0, rel_time_0 = get_sum_area_PE_in_time_window(
    clean_data.peak_start_time_s, 
    clean_data.peak_rel_start_time_s,
    clean_data.event_start_time_s, 
    clean_data.channel, 
    clean_data.board, 
    clean_data.peak_area_PE, 
    clean_data.event_id,
    time_window_width_s = 0.0000001,  # 1 us
    coincidence=3)

In [ ]:
plt.hist(sum_area_PE_list_0, bins=100
         , range=[-0.1,3000]
)

plt.hist(sum_area_PE_list, bins=100
         , range=[-0.1,3000]
)


plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
plt.yscale('log')

#### Proxy(Event Rate) for both boards

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
        #  , range = [49,51]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
        #  , range = [49,51]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
         , range = [32,34]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
         , range = [32,34]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
plt.hist(rel_time_0, bins=100
)
         
#log y
plt.yscale('log')

In [ ]:
# double check...

plt.close()
fig, ax = plt.subplots(figsize=(10,6))

# masking
for channel in range(24):

    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # this also remove null values from the peak_height_V_array
    array = list(singl_channel_data.peak_start_time_s_array)
    peak_start_time = np.concatenate(array)

    ax.hist(peak_start_time, range = [10.015,10.020], bins = 400, alpha=0.5, label=f"Channel {channel}", stacked=True,)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend()


plt.show()

### Time coincidence

In [ ]:
# double check...



plt.close()
fig, ax = plt.subplots(figsize=(10,6))

result_list = []

# masking
for channel in range(24):

    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # this also remove null values from the peak_height_V_array
    array = list(singl_channel_data.peak_start_time_s_array)
    peak_start_time = np.concatenate(array)

    np.hist(peak_start_time, range = [10.015,10.020], bins = 400, alpha=0.5, label=f"Channel {channel}", stacked=True,)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend()


plt.show()

### Test

In [ ]:
event_id = 3000
single_waveform = waveform[event_id,:]

baseline, baseline_std = get_baseline_for_all_events(waveform)
single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

event_time_s = waveform_processor.event_time_s[event_id]


In [ ]:
plt.scatter(points,single_waveform[points], color='r', label='boundaries of peaks', zorder=10)
plt.plot(single_waveform, label='filtered waveform')
plt.plot(smooth_waveform, label='waveform with rolling window')
plt.hlines(threshold, 0, 1000, 'black', linestyles='--', label='threshold')

plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')

In [ ]:
pairs = np.array([[15, 27], [38, 50], [60, 80]])
peak_width = pairs[:,1] - pairs[:,0]
tmp = np.where(peak_width < min_peak_width_sample)

pairs = np.delete(pairs, tmp, axis=0)
pairs

In [ ]:
min_peak_width_sample

In [ ]:
v_get_waveform = np.vectorize(
    get_peaks, 
    excluded=['window_size', 'threshold_sig', 'peak_width_sample'], 
    signature="(n) -> ()")

In [ ]:
single_waveform.shape

#### peak finding (nd version)

In [ ]:
axis = 1
window_size = 6
threshold_sig = 3
event_id = 3000

# single_waveform = waveform[event_id,:]
# single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

smooth_waveform = rolling_window(waveform, window_size, axis=1)
    
# add rolling window to smooth the data, roll every 3 points
# test_averaged = np.convolve(waveform, np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
# baseline, baseline_std = get_baseline_for_all_events(smooth_waveform)
threshold = baseline + threshold_sig * baseline_std

threshold = np.repeat(threshold, smooth_waveform.shape[1]).reshape(smooth_waveform.shape[0], -1)
mask = smooth_waveform > threshold

# find where the waveform passes the threshold
diff = np.diff(mask, axis = 1)

points = np.where(diff == 1)
event_id, array_idx, count = np.unique(points[0], return_counts=True, return_index=True)

# remove the last point if it's odd
odd_points = np.where(count % 2 != 0)[0]

# since all repeating points are consecutive, so we can just add the count to the index
# This will give us the end index of the last point
last_index = array_idx + count - 1
last_index_to_remove = last_index[odd_points]

# remove the last point if it's odd
peaks_position_event = np.delete(points[0], last_index_to_remove)
peaks_position_sample = np.delete(points[1], last_index_to_remove)

# double check if the peaks_position_event is odd
event_id, array_idx, count = np.unique(peaks_position_event, return_counts=True, return_index=True)
assert len(np.where(count % 2 != 0)[0]) == 0, "There are still odd points in the peaks_position_event array."


In [ ]:
test_event_id = 5000
start_idx = array_idx[test_event_id]
end_idx = array_idx[test_event_id] + count[test_event_id]
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(threshold[test_event_id], 'b--', label='threshold')
ax.plot(mask[test_event_id], 'g--', label='threshold')
ax.plot(waveform[test_event_id,:], label='smoothed waveform')
ax.plot(smooth_waveform[test_event_id,:], label='smoothed waveform')

ax.plot(peaks_position_sample[start_idx:end_idx],
        test[start_idx:end_idx], 'ro', label='start of peak')

In [ ]:
event_id = np.random.randint(0, len(baseline), size=1)
event_id

#### peak finding (1d version)

In [ ]:
event_id = np.random.randint(0, len(baseline), size=1)[0]

single_processed_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=5
extend_sum_window=50
        
window_size = 6
min_peak_width_sample = window_size*3

smooth_waveform = np.convolve(
    single_processed_waveform, 
    np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
threshold = single_baseline + threshold_sig * single_baseline_std
print(threshold)


# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask)

samples_above_threshold = np.where(diff == 1)[0]
# start samples_above_threshold are the samples_above_threshold after the rising edge
# samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

# remove the last point if it's odd
if len(samples_above_threshold) % 2 != 0:
    samples_above_threshold = samples_above_threshold[:-1]  
    
# reshape the points into pairs for better handling
peak_boundaries = samples_above_threshold.reshape(-1, 2)

# remove the pair if they are too close to each other
peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
tmp = np.where(peak_width < min_peak_width_sample)
peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)



fig, ax = plt.subplots(figsize=(10,6))
print(f"Threshold: {threshold:.3f} V")
# ax.plot(mask, 'g--', label='threshold')
ax.plot(single_processed_waveform, label='raw waveform')
ax.plot(smooth_waveform, label='smoothed waveform')
ax.axhline(threshold, color='black', linestyle = 'dashed', label='threshold')


for peak_id, peak_boundary in enumerate(peak_boundaries):
    while (extend_sum_window > 0):
        start_sample = np.max([peak_boundary[0]-extend_sum_window, 0])
        end_sample = np.min([peak_boundary[1]+extend_sum_window, len(single_processed_waveform)])

        # remove the pair if they are too far from the baseline
        y_diff = abs(single_processed_waveform[start_sample] - single_processed_waveform[end_sample])
        if y_diff > threshold_sig*single_baseline_std:
            extend_sum_window -= 10
        else:
            break
    else:
        # if we cannot find a valid window, just use the original peak boundary
        start_sample = peak_boundary[0]
        end_sample = peak_boundary[1]

    ax.plot([start_sample,end_sample],
        [single_processed_waveform[start_sample],single_processed_waveform[end_sample]], 
        'o', label = f"Peak {peak_id}")

plt.title(f"Event {event_id}")
plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')
plt.show()

In [ ]:
y_diff = abs(single_processed_waveform[start_sample] - single_processed_waveform[end_sample])
y_diff

### Test

In [ ]:
axis = 1
window_size = 4
test = np.ones((99,200))
# rolled_array = np.lib.stride_tricks.sliding_window_view(test, 4, axis=1).mean(axis=1)
rolled_array = np.lib.stride_tricks.sliding_window_view(test, window_size, axis=axis)
test_mean = rolled_array.mean(axis=test.ndim) # this is the same as the rolling mean
# print(rolled_array)
print(test_mean.shape)
# rolled_array

### Old waveform check 

In [ ]:
# can specify the event_id here
# event_id = 16641
# event_id = 8623
event_id = 1016
# event_id = None

if event_id is None:
    event_id_id = np.random.randint(0, len(event_id_array), size=1)[0]
    event_id = event_id_array[event_id_id]

print(event_id)

single_processed_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=3
extend_sum_window=50
peak_merge_window_sample = 250 # samples
        
window_size = 6
min_peak_width_sample = window_size*3

smooth_waveform = np.convolve(
    single_processed_waveform, 
    np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
threshold = single_baseline + threshold_sig * single_baseline_std
# threshold = 0.015

# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask)

samples_above_threshold = np.where(diff == 1)[0]
# start samples_above_threshold are the samples_above_threshold after the rising edge
# samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

# remove the last point if it's odd
if len(samples_above_threshold) % 2 != 0:
    samples_above_threshold = samples_above_threshold[:-1]  
    
# reshape the points into pairs for better handling
peak_boundaries = samples_above_threshold.reshape(-1, 2)

# remove the pair if they are too close to each other
peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
tmp = np.where(peak_width < min_peak_width_sample)
peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)

ax.plot(single_processed_waveform, label='waveform')
ax.axhline(single_baseline, color='green', linestyle = 'dashdot', label='baseline')
ax.axhline(threshold, color='black', linestyle = 'dashed', label='threshold')





# ax.plot(smooth_waveform, label='2. smoothed waveform')
# ax.scatter(peak_boundaries, single_processed_waveform[peak_boundaries],  s = 50, marker = '*', color = 'black', label='3. detected peaks above threshold', zorder = 10)
# print(f"Found {len(peak_boundaries)} peaks in event {event_id}.")


# avoid overlapping peaks
end_sample_of_previous_peak = 0

for peak_id, peak_boundary in enumerate(peak_boundaries):

    # find the first sample below the baseline before the peak boundary
    start_sample = peak_boundary[0] - np.where(single_processed_waveform[peak_boundary[0]::-1] - single_baseline < 0)[0][0]
    # find the first sample above the baseline after the peak boundary
    end_sample = peak_boundary[1] + np.where(single_processed_waveform[peak_boundary[1]:] - single_baseline < 0)[0][0]
    
    # skip if the start sample is before the end sample of the previous peak
    if start_sample < end_sample_of_previous_peak:
        continue

    end_sample_of_previous_peak = end_sample    
    
    peak_area_Vsample = np.sum(single_processed_waveform[start_sample:end_sample])
    peak_area_Vns = peak_area_Vsample * 4  # convert to V*ns, assuming 250 MHz sampling rate (4 ns per sample)  
    peak_area_PE = peak_area_Vns / single_info.spe_position
    peak_width_ns = (end_sample - start_sample)*4
    peak_height_V = np.max(single_processed_waveform[start_sample:end_sample])

    print(f"Peak {peak_id}:"
          f"peak_height_V = {peak_height_V:.3f}, peak_area_PE = {peak_area_PE:.3f}, "
          f"peak_width_ns = {peak_width_ns:.3f} \n")

    # ax.plot([start_sample,end_sample],
    #     [single_processed_waveform[start_sample],single_processed_waveform[end_sample]], 
    #     'o', label = f"Peak {peak_id}, area[PE] = {peak_area_PE:.1f}")
    
    # plot the found peaks in shaded area
    ax.fill_between(np.arange(start_sample, end_sample),
                    single_processed_waveform[start_sample:end_sample],
                      alpha=0.5, label=f"area = {peak_area_PE:.1f} PE")



# # merge peaks within the peak_merge_window
# n_peaks = len(peak_boundaries)
# for iterations in np.arange(0, n_peaks-1):
#     if len(peak_boundaries) < 2:
#         break

#     if len(peak_boundaries) <= iterations + 1:
#         ax.axvspan(peak_boundaries[iterations,0], peak_boundaries[iterations,1],
#                 # np.min(single_processed_waveform),
#                 # np.max(single_processed_waveform),
#                 color='grey',
#                 alpha=0.5, label=f"merge peak", zorder=1)
#         break

#     peak_window_boundary = peak_boundaries[iterations,0] + peak_merge_window_sample
#     idxs = np.where(peak_boundaries[1:,0] < peak_window_boundary)[0] # idxs is shifted by 1 due to the slicing
#     if len(idxs) == 0:
#         end_boundary = peak_boundaries[iterations,1]
#     else:
#         end_boundary = peak_boundaries[idxs[-1]+1,1]
#         peak_boundaries = np.delete(peak_boundaries, idxs+1, axis=0)
#         peak_boundaries[iterations,1] = end_boundary
#     ax.axvspan(peak_boundaries[iterations,0], end_boundary,
#                 # np.min(single_processed_waveform),
#                 # np.max(single_processed_waveform),
#                 color='grey',
#                 alpha=0.5, label=f"merge peak", zorder=1)


# merge peaks within the peak_merge_window
n_peaks = len(peak_boundaries)

for iterations in np.arange(0, n_peaks-1):
    if len(peak_boundaries) < 2:
        break

    if len(peak_boundaries) <= iterations + 1:
        break

    peak_window_boundary = peak_boundaries[iterations,0] + peak_merge_window_sample
    idxs = np.where(peak_boundaries[iterations + 1:,0] < peak_window_boundary)[0] # idxs is shifted by 1 due to the slicing
    if len(idxs) > 0:
        end_boundary = peak_boundaries[iterations + 1 + idxs[-1],1]
        peak_boundaries = np.delete(peak_boundaries, idxs+1, axis=0)
        peak_boundaries[iterations,1] = end_boundary
    ax.axvspan(peak_boundaries[iterations,0], end_boundary,
                # np.min(single_processed_waveform),
                # np.max(single_processed_waveform),
                color='grey',
                alpha=0.5, label=f"merge peak", zorder=1)

plt.title(f"Event {event_id}")
plt.legend(fontsize='15', bbox_to_anchor=(0, 1), loc='upper left')
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')

fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/example_waveform-{event_id}.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
